In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import statsmodels.api as sm

In [7]:
# Loading data
spread_df = pd.read_csv("t0_panel_BTC_spread.csv")
volume_df = pd.read_csv("t0_panel_BTC_volume.csv")
n_trades_df = pd.read_csv("t0_panel_BTC_n_trades.csv")
depth_df = pd.read_csv("t0_panel_BTC_depth.csv")

# Choosing [-600, 600] as event window
spread_df = spread_df[(spread_df["time"] >= -600) & (spread_df["time"] <= 600)]
volume_df = volume_df[(volume_df["time"] >= -600) & (volume_df["time"] <= 600)]
n_trades_df = n_trades_df[(n_trades_df["time"] >= -600) & (n_trades_df["time"] <= 600)]
depth_df = depth_df[(depth_df["time"] >= -600) & (depth_df["time"] <= 600)]

# Multiply all four variables by 100000 before computing mean
spread_df.iloc[:, 1:] *= 100000

# Calculating the average of all events at each time point
spread = spread_df.set_index("time").mean(axis=1)
volume = volume_df.set_index("time").mean(axis=1)
n_trades = n_trades_df.set_index("time").mean(axis=1)
depth = depth_df.set_index("time").mean(axis=1)

# Compute event *before* mean (time < 0) for volume, n_trades, and depth
volume_avg_pre = volume[volume.index < 0].mean()
n_trades_avg_pre = n_trades[n_trades.index < 0].mean()
depth_avg_pre = depth[depth.index < 0].mean()

# Creating post_event variable (1 after the event, 0 before)
post_event = (spread.index >= 0).astype(int)

# Constructing the regression dataset
df = pd.DataFrame({
    "spread": spread,
    "post_event": post_event,
    "volume_fixed": volume_avg_pre,   # Use pre-event mean as fixed value
    "n_trades_fixed": n_trades_avg_pre,  # Use pre-event mean as fixed value
    "depth_fixed": depth_avg_pre   # Use pre-event mean as fixed value
}).dropna()  # Remove missing values

# Setting the dependent variable (Y) and independent variable (X)
Y = df["spread"]
X = df[["post_event", "volume_fixed", "n_trades_fixed", "depth_fixed"]]  # Use fixed pre-event values
X = sm.add_constant(X, has_constant='add')  # add intercept item

# Running OLS regression
model = sm.OLS(Y, X).fit()

# Application of Newey-West HAC standard errors
model_hac = model.get_robustcov_results(cov_type="HAC", maxlags=3)

# Printing results
print(model_hac.summary())

                            OLS Regression Results                            
Dep. Variable:                 spread   R-squared:                       0.386
Model:                            OLS   Adj. R-squared:                  0.385
Method:                 Least Squares   F-statistic:                     3037.
Date:                Fri, 07 Mar 2025   Prob (F-statistic):               0.00
Time:                        10:39:52   Log-Likelihood:                -3796.3
No. Observations:                1201   AIC:                             7597.
Df Residuals:                    1199   BIC:                             7607.
Df Model:                           1                                         
Covariance Type:                  HAC                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const            6.01e-05   8.25e-07     72.

/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 4, but rank is 2
  warnings.warn('covariance of constraints does not have full '


In [8]:
# Loading data
spread_df = pd.read_csv("t1_panel_BTC_spread.csv")
volume_df = pd.read_csv("t1_panel_BTC_volume.csv")
n_trades_df = pd.read_csv("t1_panel_BTC_n_trades.csv")
depth_df = pd.read_csv("t1_panel_BTC_depth.csv")

# Choosing [-600, 600] as event window
spread_df = spread_df[(spread_df["time"] >= -600) & (spread_df["time"] <= 600)]
volume_df = volume_df[(volume_df["time"] >= -600) & (volume_df["time"] <= 600)]
n_trades_df = n_trades_df[(n_trades_df["time"] >= -600) & (n_trades_df["time"] <= 600)]
depth_df = depth_df[(depth_df["time"] >= -600) & (depth_df["time"] <= 600)]

# Multiply all four variables by 100000 before computing mean
spread_df.iloc[:, 1:] *= 100000

# Calculating the average of all events at each time point
spread = spread_df.set_index("time").mean(axis=1)
volume = volume_df.set_index("time").mean(axis=1)
n_trades = n_trades_df.set_index("time").mean(axis=1)
depth = depth_df.set_index("time").mean(axis=1)

# Compute event *before* mean (time < 0) for volume, n_trades, and depth
volume_avg_pre = volume[volume.index < 0].mean()
n_trades_avg_pre = n_trades[n_trades.index < 0].mean()
depth_avg_pre = depth[depth.index < 0].mean()

# Creating post_event variable (1 after the event, 0 before)
post_event = (spread.index >= 0).astype(int)

# Constructing the regression dataset
df = pd.DataFrame({
    "spread": spread,
    "post_event": post_event,
    "volume_fixed": volume_avg_pre,   # Use pre-event mean as fixed value
    "n_trades_fixed": n_trades_avg_pre,  # Use pre-event mean as fixed value
    "depth_fixed": depth_avg_pre   # Use pre-event mean as fixed value
}).dropna()  # Remove missing values

# Setting the dependent variable (Y) and independent variable (X)
Y = df["spread"]
X = df[["post_event", "volume_fixed", "n_trades_fixed", "depth_fixed"]]  # Use fixed pre-event values
X = sm.add_constant(X, has_constant='add')  # add intercept item

# Running OLS regression
model = sm.OLS(Y, X).fit()

# Application of Newey-West HAC standard errors
model_hac = model.get_robustcov_results(cov_type="HAC", maxlags=3)

# Printing results
print(model_hac.summary())

                            OLS Regression Results                            
Dep. Variable:                 spread   R-squared:                       0.193
Model:                            OLS   Adj. R-squared:                  0.192
Method:                 Least Squares   F-statistic:                     2076.
Date:                Fri, 07 Mar 2025   Prob (F-statistic):               0.00
Time:                        10:39:56   Log-Likelihood:                -3763.2
No. Observations:                1201   AIC:                             7530.
Df Residuals:                    1199   BIC:                             7541.
Df Model:                           1                                         
Covariance Type:                  HAC                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const           7.805e-05   1.29e-06     60.

/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 4, but rank is 2
  warnings.warn('covariance of constraints does not have full '


In [ ]:
=========================================================================================================================

In [25]:
# Loading data
spread_df = pd.read_csv("panel_BTC_spread.csv")
volume_df = pd.read_csv("panel_BTC_volume.csv")
n_trades_df = pd.read_csv("panel_BTC_n_trades.csv")
depth_df = pd.read_csv("panel_BTC_depth.csv")

# Choosing [-600, 600] as event window
spread_df = spread_df[(spread_df["time"] >= -600) & (spread_df["time"] <= 600)]
volume_df = volume_df[(volume_df["time"] >= -600) & (volume_df["time"] <= 600)]
n_trades_df = n_trades_df[(n_trades_df["time"] >= -600) & (n_trades_df["time"] <= 600)]
depth_df = depth_df[(depth_df["time"] >= -600) & (depth_df["time"] <= 600)]

# Multiply spread by 100000 before computing mean
spread_df.iloc[:, 1:] *= 100000

# Calculating the average of all events at each time point
spread = spread_df.set_index("time").mean(axis=1)
volume = volume_df.set_index("time").mean(axis=1)
n_trades = n_trades_df.set_index("time").mean(axis=1)
depth = depth_df.set_index("time").mean(axis=1)

# Creating post_event variable (1 after the event, 0 before)
post_event = (spread.index >= 0).astype(int)

# Constructing the regression dataset
df = pd.DataFrame({
    "spread": spread,
    "post_event": post_event,
    "volume_fixed": volume,
    "n_trades_fixed": n_trades,
    "depth_fixed": depth
}).dropna()  # Remove missing values

# Setting the dependent variable (Y) and independent variable (X)
Y = df["spread"]
X = df[["post_event", "volume_fixed", "n_trades_fixed", "depth_fixed"]]  # Change these variables to do robust test
X = sm.add_constant(X, has_constant='add')  # add intercept item

# Running OLS regression
model = sm.OLS(Y, X).fit()

# Application of Newey-West HAC standard errors
model_hac = model.get_robustcov_results(cov_type="HAC", maxlags=3)

# Printing results
print(model_hac.summary())

                            OLS Regression Results                            
Dep. Variable:                 spread   R-squared:                       0.876
Model:                            OLS   Adj. R-squared:                  0.876
Method:                 Least Squares   F-statistic:                     343.0
Date:                Fri, 07 Mar 2025   Prob (F-statistic):          1.11e-196
Time:                        11:19:14   Log-Likelihood:                -2611.9
No. Observations:                1201   AIC:                             5234.
Df Residuals:                    1196   BIC:                             5259.
Df Model:                           4                                         
Covariance Type:                  HAC                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.0579      0.827      0.

In [26]:
# Loading data
spread_df = pd.read_csv("panel_DOGE_spread.csv")
volume_df = pd.read_csv("panel_DOGE_volume.csv")
n_trades_df = pd.read_csv("panel_DOGE_n_trades.csv")
depth_df = pd.read_csv("panel_DOGE_depth.csv")

# Choosing [-600, 600] as event window
spread_df = spread_df[(spread_df["time"] >= -600) & (spread_df["time"] <= 600)]
volume_df = volume_df[(volume_df["time"] >= -600) & (volume_df["time"] <= 600)]
n_trades_df = n_trades_df[(n_trades_df["time"] >= -600) & (n_trades_df["time"] <= 600)]
depth_df = depth_df[(depth_df["time"] >= -600) & (depth_df["time"] <= 600)]

# Multiply spread by 100000 before computing mean
spread_df.iloc[:, 1:] *= 100000

# Calculating the average of all events at each time point
spread = spread_df.set_index("time").mean(axis=1)
volume = volume_df.set_index("time").mean(axis=1)
n_trades = n_trades_df.set_index("time").mean(axis=1)
depth = depth_df.set_index("time").mean(axis=1)

# Creating post_event variable (1 after the event, 0 before)
post_event = (spread.index >= 0).astype(int)

# Constructing the regression dataset
df = pd.DataFrame({
    "spread": spread,
    "post_event": post_event,
    "volume_fixed": volume,
    "n_trades_fixed": n_trades,
    "depth_fixed": depth
}).dropna()  # Remove missing values

# Setting the dependent variable (Y) and independent variable (X)
Y = df["spread"]
X = df[["post_event", "volume_fixed", "n_trades_fixed", "depth_fixed"]]  # Change these variables to do robust test
X = sm.add_constant(X, has_constant='add')  # add intercept item

# Running OLS regression
model = sm.OLS(Y, X).fit()

# Application of Newey-West HAC standard errors
model_hac = model.get_robustcov_results(cov_type="HAC", maxlags=3)

# Printing results
print(model_hac.summary())

                            OLS Regression Results                            
Dep. Variable:                 spread   R-squared:                       0.843
Model:                            OLS   Adj. R-squared:                  0.843
Method:                 Least Squares   F-statistic:                     365.8
Date:                Fri, 07 Mar 2025   Prob (F-statistic):          9.70e-206
Time:                        11:21:20   Log-Likelihood:                -3055.9
No. Observations:                1201   AIC:                             6122.
Df Residuals:                    1196   BIC:                             6147.
Df Model:                           4                                         
Covariance Type:                  HAC                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const             46.7952      0.877     53.

In [27]:
# Loading data
spread_df = pd.read_csv("panel_SHIB_spread.csv")
volume_df = pd.read_csv("panel_SHIB_volume.csv")
n_trades_df = pd.read_csv("panel_SHIB_n_trades.csv")
depth_df = pd.read_csv("panel_SHIB_depth.csv")

# Choosing [-600, 600] as event window
spread_df = spread_df[(spread_df["time"] >= -600) & (spread_df["time"] <= 600)]
volume_df = volume_df[(volume_df["time"] >= -600) & (volume_df["time"] <= 600)]
n_trades_df = n_trades_df[(n_trades_df["time"] >= -600) & (n_trades_df["time"] <= 600)]
depth_df = depth_df[(depth_df["time"] >= -600) & (depth_df["time"] <= 600)]

# Multiply spread by 100000 before computing mean
spread_df.iloc[:, 1:] *= 100000

# Calculating the average of all events at each time point
spread = spread_df.set_index("time").mean(axis=1)
volume = volume_df.set_index("time").mean(axis=1)
n_trades = n_trades_df.set_index("time").mean(axis=1)
depth = depth_df.set_index("time").mean(axis=1)

# Creating post_event variable (1 after the event, 0 before)
post_event = (spread.index >= 0).astype(int)

# Constructing the regression dataset
df = pd.DataFrame({
    "spread": spread,
    "post_event": post_event,
    "volume_fixed": volume,
    "n_trades_fixed": n_trades,
    "depth_fixed": depth
}).dropna()  # Remove missing values

# Setting the dependent variable (Y) and independent variable (X)
Y = df["spread"]
X = df[["post_event", "volume_fixed", "n_trades_fixed", "depth_fixed"]]  # Change these variables to do robust test
X = sm.add_constant(X, has_constant='add')  # add intercept item

# Running OLS regression
model = sm.OLS(Y, X).fit()

# Application of Newey-West HAC standard errors
model_hac = model.get_robustcov_results(cov_type="HAC", maxlags=3)

# Printing results
print(model_hac.summary())

                            OLS Regression Results                            
Dep. Variable:                 spread   R-squared:                       0.832
Model:                            OLS   Adj. R-squared:                  0.831
Method:                 Least Squares   F-statistic:                     309.9
Date:                Fri, 07 Mar 2025   Prob (F-statistic):          3.73e-109
Time:                        11:22:09   Log-Likelihood:                -2874.7
No. Observations:                1201   AIC:                             5759.
Df Residuals:                    1196   BIC:                             5785.
Df Model:                           4                                         
Covariance Type:                  HAC                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const            108.0033      1.883     57.

/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 4, but rank is 2
  warnings.warn('covariance of constraints does not have full '


In [28]:
# Loading data
spread_df = pd.read_csv("panel_USDT_spread.csv")
volume_df = pd.read_csv("panel_USDT_volume.csv")
n_trades_df = pd.read_csv("panel_USDT_n_trades.csv")
depth_df = pd.read_csv("panel_USDT_depth.csv")

# Choosing [-600, 600] as event window
spread_df = spread_df[(spread_df["time"] >= -600) & (spread_df["time"] <= 600)]
volume_df = volume_df[(volume_df["time"] >= -600) & (volume_df["time"] <= 600)]
n_trades_df = n_trades_df[(n_trades_df["time"] >= -600) & (n_trades_df["time"] <= 600)]
depth_df = depth_df[(depth_df["time"] >= -600) & (depth_df["time"] <= 600)]

# Multiply spread by 100000 before computing mean
spread_df.iloc[:, 1:] *= 100000

# Calculating the average of all events at each time point
spread = spread_df.set_index("time").mean(axis=1)
volume = volume_df.set_index("time").mean(axis=1)
n_trades = n_trades_df.set_index("time").mean(axis=1)
depth = depth_df.set_index("time").mean(axis=1)

# Creating post_event variable (1 after the event, 0 before)
post_event = (spread.index >= 0).astype(int)

# Constructing the regression dataset
df = pd.DataFrame({
    "spread": spread,
    "post_event": post_event,
    "volume_fixed": volume,
    "n_trades_fixed": n_trades,
    "depth_fixed": depth
}).dropna()  # Remove missing values

# Setting the dependent variable (Y) and independent variable (X)
Y = df["spread"]
X = df[["post_event", "volume_fixed", "n_trades_fixed", "depth_fixed"]]  # Change these variables to do robust test
X = sm.add_constant(X, has_constant='add')  # add intercept item

# Running OLS regression
model = sm.OLS(Y, X).fit()

# Application of Newey-West HAC standard errors
model_hac = model.get_robustcov_results(cov_type="HAC", maxlags=3)

# Printing results
print(model_hac.summary())

                            OLS Regression Results                            
Dep. Variable:                 spread   R-squared:                       0.222
Model:                            OLS   Adj. R-squared:                  0.219
Method:                 Least Squares   F-statistic:                     14.41
Date:                Fri, 07 Mar 2025   Prob (F-statistic):           1.69e-11
Time:                        11:22:54   Log-Likelihood:                -2003.3
No. Observations:                1201   AIC:                             4017.
Df Residuals:                    1196   BIC:                             4042.
Df Model:                           4                                         
Covariance Type:                  HAC                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const              4.4525      0.934      4.